# SQL Integration and Business Reporting
Connect to PostgreSQL, extract with parameterized SQL, and visualize sales trends.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.db_utils import DatabaseManager

In [ ]:
db = DatabaseManager()
db.query_to_df('SELECT current_database() AS database, now() AS connected_at')

## Parameterized extraction
User-controlled dates are bound as parameters rather than interpolated into SQL.

In [ ]:
monthly_sales = db.query_to_df('''
    SELECT date_trunc('month', o.order_date)::date AS month,
           SUM(oi.quantity * oi.unit_price) AS revenue
    FROM ecommerce.orders o
    JOIN ecommerce.order_items oi USING (order_id)
    WHERE o.status = :status AND o.order_date >= :start_date
    GROUP BY 1 ORDER BY 1
''', {'status': 'completed', 'start_date': '2024-01-01'})
monthly_sales

In [ ]:
monthly_sales['month'] = monthly_sales['month'].astype('datetime64[ns]')
monthly_sales.plot(x='month', y='revenue', kind='line', marker='o', legend=False, title='Completed monthly revenue')
plt.ylabel('Revenue')
plt.tight_layout()
plt.show()

In [ ]:
top_customers = db.query_to_df('''
    SELECT * FROM ecommerce.customer_revenue
    ORDER BY revenue DESC
    LIMIT :limit
''', {'limit': 10})
top_customers

Close the pooled connections when the notebook session is finished.

In [ ]:
db.close()